# Trabajo Práctico N°3 — Minería de Datos

> **Año:** 2026<br>
> **Materia:** Minería de Datos — Tecnicatura Universitaria en Inteligencia Artificial (FCEIA, UNR)<br>
> **Integrantes:**
> * Larrubia, Agustín.
> * Lazarte, Manuel.

## Introducción:
&emsp;El objetivo de este trabajo práctico es integrar los conocimientos adquiridos en las
unidades 5 y 6 en un problema real de clasificación multiclase, asociado a la
recomendación de cultivos agrícolas en función de parámetros de suelo y clima los cuales encontramos en el dataset 'crop_recommendation_geo.csv'. La
variable objetivo es Clase (tipo de cultivo recomendado, con 10 clases).

## Descripción del conjunto de datos

El conjunto de datos `crop_recommendation_geo.csv` contiene **50 004 registros** y **17 columnas** (el enunciado menciona 50 005 porque cuenta la línea de encabezado). Fue **modificado intencionalmente** respecto del original ([Kaggle - Global Crop Recommendation](https://www.kaggle.com/datasets/markjehan/global-croprecommendation)).

### Variables predictoras (16)

| Variable | Descripción |
|---|---|
| `N` | Nitrógeno total en el suelo (0-30 cm) |
| `P` | Fósforo disponible para las plantas (Olsen) |
| `exch_bases` | Bases intercambiables totales (Ca + Mg + K + Na), capa superficial del suelo: la medición de potasio |
| `ph` | pH del suelo en agua (0-30 cm) |
| `soc` | Carbono orgánico del suelo (0-30 cm) |
| `arcilla` | Fracción arcillosa (0-30 cm) |
| `arena` | Fracción de arena (0-30 cm) |
| `cec` | Capacidad de intercambio catiónico |
| `temperatura` | Temperatura media anual del aire |
| `temp_min_mensual` | Temperatura media del mes más frío |
| `temp_max_mensual` | Temperatura media del mes más cálido |
| `humedad` | Humedad relativa media anual |
| `lluvia` | Precipitación anual |
| `estacionalidad_lluvias` | Coeficiente de variación de la precipitación mensual (BIO15) |
| `radiacion_solar` | Radiación solar incidente media anual |
| `elevacion` | Altura sobre el nivel del mar |

### Variable objetivo

`Clase`: cultivo recomendado, con **10 clases**: Algodón, Arroz, Banana, Café, Coco, Garbanzo, Frijol, Lenteja, Maíz y Manzana.

> **Nota:** en todo el trabajo se utiliza `random_state=13` en modelos, particiones y validaciones cruzadas para garantizar la reproducibilidad.

# Librerías y configuración

In [ ]:
# MANIPULACIÓN DE DATOS
import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

# VISUALIZACIÓN
import matplotlib.pyplot as plt
import seaborn as sns

# PREPARACIÓN DE LOS DATOS
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split

# MODELOS
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# MÉTRICAS
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, ConfusionMatrixDisplay)

# CONFIGURACIÓN GENERAL
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

# Semilla única para modelos, particiones y validaciones cruzadas (consigna del TP).
SEMILLA = 13

import sklearn
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

In [ ]:
def target_balance(y):
    """Balance de clases (adaptado del notebook de SVM de la cátedra)."""
    cantidad = y.value_counts()
    porcentaje = y.value_counts(normalize=True) * 100
    return pd.DataFrame({"Cantidad": cantidad, "Porcentaje": porcentaje.round(2)})


def outliers_iqr(s):
    """Máscara booleana de los valores fuera de [Q1 - 1.5·IQR, Q3 + 1.5·IQR]."""
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

## 1. Preparación de datos

In [ ]:
# El CSV está en la carpeta crop_recommendation/ del repositorio.
# Si se ejecuta en Colab con el archivo subido a la raíz, se usa esa ruta.
RUTA_CSV = Path("crop_recommendation/crop_recommendation_geo.csv")
if not RUTA_CSV.exists():
    RUTA_CSV = Path("crop_recommendation_geo.csv")

df_original = pd.read_csv(RUTA_CSV, encoding="utf-8-sig", sep=";")
print("Dimensiones:", df_original.shape)
df_original.head()

### 1.1 Estructura y tipos de datos

In [ ]:
df = df_original.copy()

# Normalizamos los nombres de las columnas (sin espacios, en minúscula)
df.columns = df.columns.str.strip().str.lower()

print(df.dtypes)

Salvo `lluvia` y `elevacion`, todas las variables numéricas se leyeron como texto porque el archivo usa **coma decimal**. Las convertimos a `float`.

In [ ]:
num_cols = [
    'n', 'p', 'exch_bases', 'ph', 'soc', 'arcilla', 'arena', 'cec',
    'temperatura', 'temp_min_mensual', 'temp_max_mensual', 'humedad',
    'lluvia', 'estacionalidad_lluvias', 'radiacion_solar', 'elevacion'
]
cols_texto = [c for c in num_cols if not pd.api.types.is_numeric_dtype(df[c])]

for col in cols_texto:
    df[col] = df[col].str.replace(',', '.', regex=False).astype(float)

print("Columnas convertidas:", cols_texto)
print("\nClases presentes:", sorted(df['clase'].unique()))
df.dtypes.value_counts()

### 1.2 Análisis exploratorio de datos (EDA)

El EDA se hace sobre los datos **crudos** (antes de limpiar), para que los problemas del dataset queden visibles.

#### 1.2.1 Distribuciones

In [ ]:
df[num_cols].describe().T

**Hipótesis:**

1. Como en el `describe()` vimos máximos imposibles en `ph` (25,3) y `humedad` (148 %), esperamos que sus histogramas tengan el eje x estirado mucho más allá de los valores normales, con el grueso de los datos concentrado en una zona angosta y uno o pocos valores aislados lejos. No esperamos una cola larga, porque se trata de casos puntuales y la media y la mediana son parecidas. El tratamiento de estos valores se hace en la sección 1.3.
2. Como el dataset reúne 10 cultivos con requerimientos climáticos y de suelo distintos, esperamos que `lluvia`, `temperatura`, `temp_min_mensual`, `temp_max_mensual`, `p` y `soc` muestren varios picos (distribuciones multimodales), cada uno asociado a grupos de cultivos. Si alguna de estas variables muestra un único pico, eso podría indicar que no distingue tanto entre cultivos.
3. Las variables cuya media es bastante mayor que la mediana en el `describe()` (por ejemplo `elevacion`, `lluvia`, `n`, `p`, `soc`) deberían estar sesgadas a la derecha: muchos registros con valores bajos y pocos con valores altos.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(data=df, x=col, kde=True, bins=30, element='step', ax=ax)
    ax.set_title(col)
plt.suptitle('Distribución de las variables predictoras (datos crudos)', fontsize=16)
plt.tight_layout()
plt.show()

**Observación:**

1. Valores imposibles (hipótesis 1): en `ph` el eje llega hasta 25, pero después de 9–10 prácticamente no hay barras. El 25,3 es un caso aislado que estira el eje, no una cola. Lo mismo ocurre en `humedad`, que llega a más de 140 sin formar ninguna "montaña". Se cumple. Además aparecieron valores imposibles en variables que no habíamos señalado en la hipótesis: `arcilla` tiene una cola mínima hasta 142 (más de 100 %), aunque su grueso se concentra alrededor de la media, casi igual a la mediana, con forma aproximadamente simétrica. También `temp_max_mensual` llega a 53 °C y `temperatura` a 44,6 °C. Son medias (mensual y anual) y superan los récords de esas medias: ≈ 42,3 °C para el mes más cálido [1] y ≈ 34–35 °C para la media anual [2]. Los récords de 54–56 °C corresponden a temperaturas puntuales, no a promedios. Además son valores aislados: los siguientes más altos son 38,2 y 30,4.
2. Varios picos (hipótesis 2): `temp_max_mensual` muestra dos picos además de una cola larga hacia la izquierda. `lluvia` 2 picos (400 mm y 1000 mm), `temperatura` un pico dominante cerca de 26 °C y varios picos secundarios, `temp_min_mensual` varios picos alrededor de −15, 20 y 25 °C, `p` 1 pico y `soc` 1 pico. Se cumple parcialmente: las variables climáticas muestran varios picos, pero `p` y `soc` no.
3. Asimetría (hipótesis 3): `n`, `p`, `soc`, `elevacion` y `lluvia` están sesgadas a la derecha. En `elevacion` la mitad de los registros está por debajo de ≈ 385 m y el 75 % por debajo de ≈ 810 m, mientras que la media es ≈ 600 m: los pocos registros de gran altura tiran la media hacia arriba. Se cumple.

**No previsto**: 
* `ph` es bimodal, con picos cerca de 5 y de 7. También muestran dos picos `humedad`, la `estacionalidad_lluvias` y la `radiacion_solar`. Una posible explicación es que el dataset mezcla cultivos de regiones con climas distintos; lo podemos chequear con los boxplots por clase.
* `temp_min_mensual` y `temperatura` muestran el caso inverso a la hipótesis 3: media menor que la mediana (11,9 vs 16,4 °C y 19,2 vs 22,0 °C, respectivamente) y cola larga hacia la izquierda (sesgo a la izquierda), debido a que la mayoría de los registros son cálidos y hay pocos muy fríos.
* `exch_bases` cuenta con una distribución multimodal con picos cerca de 5, 22 y 35.

**Para lo que viene:**
* Viendo estos gráficos y pensando en los modelos, prevemos que las variables muy sesgadas y con valores extremos afectarán más a la SVM, porque trabaja con distancias y esos extremos inflan el escalado, que al Random Forest, que solo compara cada variable contra un umbral y por eso depende únicamente del orden de los valores, no de su magnitud.
* Sospechamos que los grupos de cultivos se diferencian más por el clima que por `p` y `soc`, porque las variables climáticas muestran varios picos y esas dos muestran uno solo. Aun así, `ph` y `exch_bases`, que también son de suelo, son multimodales. Lo contrastaremos con la importancia de variables del Random Forest.

**Hipótesis:**

1. Las variables que en los histogramas se vieron sesgadas a la derecha (`n`, `p`, `soc`, `lluvia`, `elevacion`) van a tener **muchos** puntos por encima del bigote superior (del orden de miles), formando una nube continua.
2. En `ph`, `humedad`, `arcilla`, `temperatura` y `temp_max_mensual` esperamos ver uno o dos puntos **aislados y muy alejados** del resto: los valores imposibles que ya vimos en el `describe()`.
3. Las variables más simétricas (`arena`, `radiacion_solar`, `estacionalidad_lluvias`) deberían tener pocos outliers.
4. Esperamos que la gran mayoría de los outliers **no** sean errores sino valores reales de cultivos con requerimientos extremos, porque el dataset mezcla 10 cultivos de climas muy distintos.


In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
for ax, col in zip(axes.flatten(), num_cols):
    sns.boxplot(data=df, y=col, ax=ax)
    ax.set_title(col)
plt.suptitle('Boxplots de las variables predictoras (datos crudos)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
pd.DataFrame({'Outliers según IQR': {col: outliers_iqr(df[col]).sum() for col in num_cols}})

**Observación:**

1. Las variables con más outliers según IQR son `soc` (3665), `p` (3453), `elevacion` (2825), `n` (1483) y `lluvia` (1360), todos por encima del bigote superior y formando nubes densas, no puntos sueltos. **Se cumple.** Lo que no previmos es que `cec` también tuviera muchos (2162).
2. `ph` tiene exactamente 2 outliers (18 y 25,3) y `humedad` 1 (148): son justamente los valores imposibles, muy separados del resto. Lo mismo se ve en `arcilla` (142), `temperatura` (44,6) y `temp_max_mensual` (53). **Se cumple.**
3. `radiacion_solar` (41) y `estacionalidad_lluvias` (8) tienen pocos outliers, pero `arena` tiene 849. **Se cumple parcialmente.** Además, casi todos los 535 outliers de `temp_max_mensual` están por **debajo** del bigote inferior: lugares con veranos frescos (de altura o latitudes altas), que no habíamos anticipado.
4. Con el boxplot solo **no se puede** distinguir un valor imposible de uno raro pero posible: en `arcilla` el punto de 142 % aparece igual que los casi 300 outliers reales (por debajo de 10 % y entre 55 % y 70 %). El criterio IQR marca miles de valores y no es razonable pensar que todos son errores. Por eso, en 1.3.3 los errores se detectan con **reglas de dominio**, y en 1.3.4 analizamos a qué cultivos pertenecen los outliers.


#### 1.2.2 Análisis de correlaciones

**Hipótesis:**

1. Las tres temperaturas (`temperatura`, `temp_min_mensual`, `temp_max_mensual`) van a estar fuertemente correlacionadas en forma positiva, y es la única relación que podría superar |r| > 0,9.
2. `arcilla` y `arena` tendrán correlación negativa, porque son fracciones complementarias de la misma muestra de suelo.
3. `humedad` y `lluvia` tendrán correlación positiva.
4. Las temperaturas tendrán correlación negativa con `elevacion` (hace más frío en altura).
5. `soc` y `n` tendrán correlación positiva (el nitrógeno del suelo está mayormente en la materia orgánica), y `cec` positiva con `arcilla` y con `soc`.


In [ ]:
plt.figure(figsize=(14, 12))
corr = df[num_cols].corr(method='pearson')
ax = sns.heatmap(
    corr,
    vmin=-1, vmax=1, center=0,
    cmap=sns.diverging_palette(20, 220, n=200),
    square=True,
    annot=True, fmt='.2f',
    annot_kws={'size': 8}
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
plt.title('Matriz de correlación (Pearson)', fontsize=16)
plt.tight_layout()
plt.show()

**Observación:**

1. `temperatura` y `temp_min_mensual` tienen r = 0,95 (la única por encima de 0,9); `temperatura` con `temp_max_mensual`, 0,75; y las dos mensuales entre sí, 0,52. **Se cumple.**
2. `arcilla` y `arena`: r = −0,65. **Se cumple.**
3. `humedad` y `lluvia`: r = 0,66. **Se cumple.**
4. `elevacion` se correlaciona negativamente con `temp_max_mensual` (−0,65) y con `temperatura` (−0,43). **Se cumple.**
5. `soc` y `n`: r = 0,54, y `cec` con `arcilla`: 0,42. Pero `cec` con `soc` es prácticamente nula (0,06). **Se cumple parcialmente.**

**No previsto:** las correlaciones más fuertes después de `temperatura`–`temp_min_mensual` (0,95) son entre `ph` y `humedad` (−0,78), `ph` y `lluvia` (−0,77) y `ph` y `exch_bases` (0,71): en zonas lluviosas el suelo es más ácido y tiene menos bases intercambiables, porque la lluvia las lava. También `radiacion_solar` acompaña a las temperaturas (0,60–0,69).

Esta matriz mide relaciones entre **predictoras**, no dice qué variable separa mejor las **clases**; eso lo responde la importancia de variables del Random Forest (4.3). Los 2 registros imposibles casi no la afectan: calculada sobre los datos limpios, `ph`–`humedad` pasa de −0,78 a −0,79. Con 50 000 filas, un par de valores extremos mueven la correlación en la segunda cifra decimal como mucho.


#### 1.2.3 Distribución de la variable objetivo (`clase`)

**Hipótesis:** esperamos una distribución casi balanceada, con alrededor de 5000 registros por clase (50 004 / 10). Si alguna clase tiene unos pocos registros "de más", podrían ser filas agregadas al modificar el dataset (por ejemplo, las que tienen errores).


In [ ]:
target_balance(df["clase"])

**Observación:** las clases tienen entre 5000 y 5002 registros, es decir, el 10 % cada una. **Se cumple.** Las únicas con registros de más son `cafe` (+2), `arroz` (+1) y `manzana` (+1). Al final de 1.3 se ve que los 2 de más de `cafe` coinciden con las 2 filas de café con errores (el faltante y el pH de 25,3), y que entre los eliminados de `manzana` está la fila con valores imposibles. Con este balance, un promedio *weighted* da prácticamente lo mismo que uno *macro*, y la estratificación de los folds es suficiente. El desbalance real aparece **después** de la limpieza (`lenteja` queda con 4885 y `banana` con 5000), y aun así es menor al 3 %.


### 1.3 Detección y tratamiento de problemas del dataset

Orden de la limpieza: **(1)** valores faltantes → **(2)** duplicados exactos → **(3)** registros con las mismas variables pero distinta clase → **(4)** valores físicamente imposibles → **(5)** análisis de outliers. Cada paso muestra con código qué se encontró antes de eliminarlo.

In [ ]:
# Guardamos los conteos por clase para comparar al final de la limpieza.
conteo_inicial = df['clase'].value_counts()
n_inicial = len(df)

#### 1.3.1 Valores faltantes

In [ ]:
faltantes = df.isna().sum()
print("Valores faltantes por columna:")
print(faltantes[faltantes > 0])

print("\nFilas con al menos un faltante:")
df[df.isna().any(axis=1)]

**Decisión: eliminar.** Los 3 valores faltantes (`ph`, `cec`, `humedad`) están en **una sola fila**. Imputarla implicaría inventar 3 de sus 16 variables, mientras que eliminarla representa el 0,002 % del dataset.

In [ ]:
df = df.dropna()
print("Dimensiones tras eliminar faltantes:", df.shape)

#### 1.3.2 Registros duplicados

In [ ]:
duplicados = df.duplicated()
print("Duplicados exactos:", duplicados.sum())
print("\nDuplicados por clase:")
df.loc[duplicados, 'clase'].value_counts()

**Decisión: eliminar los duplicados exactos (conservando la primera aparición).** Con validación cruzada, una copia podría quedar en entrenamiento y la otra en validación del mismo fold: el modelo sería evaluado sobre un punto que ya vio, inflando las métricas.

In [ ]:
df = df.drop_duplicates(keep='first')
print("Dimensiones tras eliminar duplicados:", df.shape)

Además de los duplicados exactos, buscamos registros con **las mismas 16 variables pero distinta clase**:

In [ ]:
mismas_variables = df[df.duplicated(subset=num_cols, keep=False)]
clases_por_punto = mismas_variables.groupby(num_cols)['clase'].agg(lambda s: tuple(sorted(s.unique())))
conflictos = clases_por_punto[clases_por_punto.apply(len) > 1]

print("Puntos con etiquetas contradictorias:", len(conflictos))
print("Filas involucradas:", len(mismas_variables))
print("\nPares de clases en conflicto:")
print(conflictos.value_counts())

# Guardamos los pares para cruzarlos con las matrices de confusión (sección 5.2).
pares_conflictivos = {frozenset(par) for par in conflictos}

**Decisión: eliminar ambas filas de cada par.** Un mismo punto con dos cultivos distintos puede ser un error de etiqueta o un lugar apto para los dos. No tenemos forma de saberlo y, para un modelo que devuelve una sola clase, ese punto no tiene una respuesta correcta, así que eliminamos ambas filas. Los pares de clases involucrados se guardan para el análisis de la sección 5.2.

In [ ]:
df = df.drop_duplicates(subset=num_cols, keep=False)
print("Dimensiones tras eliminar etiquetas contradictorias:", df.shape)

**Justificación:** los duplicados exactos no se reparten parejo: `lenteja` (109) y `manzana` (55) concentran el 83 %, una concentración que no se esperaría si fueran repeticiones al azar. Las etiquetas contradictorias aparecen entre cultivos agronómicamente cercanos (`garbanzo`–`lenteja`, ambas legumbres de clima templado y seco, y `maiz`–`manzana`). Otra opción habría sido conservar una de las dos filas, pero no hay ningún criterio para saber cuál es la correcta: elegir al azar podría dejar la etiqueta equivocada. Eliminar ambas cuesta apenas 14 filas (0,03 % del dataset). Volvemos a estos pares en las matrices de confusión (5.2).


#### 1.3.3 Valores físicamente imposibles

El criterio IQR solo dice qué valores son **raros respecto del resto**, no cuáles son **imposibles**. Para detectar errores usamos **reglas de dominio**: límites que ninguna medición real puede violar.

| Regla | Fundamento |
|---|---|
| `ph` ∈ [0, 14] | Rango de la escala de pH en agua |
| `humedad` ∈ [0, 100] | Es una humedad **relativa** (%) |
| `arcilla`, `arena` ∈ [0, 100] y `arcilla + arena` ≤ 100 | Son fracciones (%) de la misma muestra de suelo; el resto es limo |
| `n`, `p`, `exch_bases`, `soc`, `cec`, `lluvia`, `estacionalidad_lluvias`, `radiacion_solar` ≥ 0 | Son cantidades o concentraciones |
| `temp_min_mensual` ≤ `temperatura` ≤ `temp_max_mensual` | La media anual está entre la media del mes más frío y la del más cálido |
| `temperatura` ≤ 35 °C | La mayor temperatura media anual registrada es ≈ 34–35 °C según la fuente (Dallol, Etiopía, 1960–1966) [2] |
| `temp_max_mensual` ≤ 43 °C | La mayor media mensual registrada es ≈ 42,3 °C (Death Valley, julio de 2018) [1]. Los récords de 54–56 °C son temperaturas puntuales, no medias |
| `elevacion` ∈ [−430, 8849] m | Costa del Mar Muerto y cima del Everest |

In [ ]:
no_negativas = ['n', 'p', 'exch_bases', 'soc', 'cec', 'lluvia',
                'estacionalidad_lluvias', 'radiacion_solar']

reglas = {
    'ph fuera de [0, 14]':                          ~df['ph'].between(0, 14),
    'humedad fuera de [0, 100]':                    ~df['humedad'].between(0, 100),
    'arcilla fuera de [0, 100]':                    ~df['arcilla'].between(0, 100),
    'arena fuera de [0, 100]':                      ~df['arena'].between(0, 100),
    'arcilla + arena > 100':                        (df['arcilla'] + df['arena']) > 100,
    'valores negativos en magnitudes >= 0':         (df[no_negativas] < 0).any(axis=1),
    'temperatura fuera de [temp_min, temp_max]':    ~df['temperatura'].between(df['temp_min_mensual'],
                                                                           df['temp_max_mensual']),
    'temperatura media anual > 35 °C':              df['temperatura'] > 35,
    'temp. media del mes más cálido > 43 °C':       df['temp_max_mensual'] > 43,
    'elevacion fuera de [-430, 8849] m':            ~df['elevacion'].between(-430, 8849),
}

resumen_reglas = pd.DataFrame({
    'Registros que la violan': {r: m.sum() for r, m in reglas.items()},
    'Índices': {r: list(df.index[m]) for r, m in reglas.items()},
})
resumen_reglas

In [ ]:
imposibles = pd.concat(reglas, axis=1).any(axis=1)
print("Registros con al menos un valor imposible:", imposibles.sum())
df[imposibles]

**Decisión: eliminar.** Son registros con varios valores que no pueden existir físicamente (por ejemplo, pH 25 o humedad relativa de 148 %). No son outliers naturales sino errores, y no hay un valor "correcto" con el cual reemplazarlos.

In [ ]:
df = df[~imposibles]
print("Dimensiones tras eliminar valores imposibles:", df.shape)

In [ ]:
# Valores extremos pero posibles: se detectan para documentarlos, pero no se eliminan.
print("Registros con P = 0 (fósforo bajo el límite de detección, posible):", (df['p'] == 0).sum())
print("Registros con elevación < 0 m (zonas bajo el nivel del mar, posible):", (df['elevacion'] < 0).sum())

In [ ]:
resumen_limpieza = pd.DataFrame({'Antes': conteo_inicial, 'Después': df['clase'].value_counts()})
resumen_limpieza['Eliminados'] = resumen_limpieza['Antes'] - resumen_limpieza['Después']
print(f"Registros: {n_inicial} → {len(df)} ({n_inicial - len(df)} eliminados)")
resumen_limpieza

#### 1.3.4 Outliers (criterio IQR)

**Hipótesis:**

1. `lluvia`: los cultivos tropicales húmedos (`coco`, `banana`, `cafe`, `arroz`) van a tener valores mucho más altos que las legumbres de clima seco (`lenteja`, `garbanzo`).
2. `soc` (carbono orgánico): más alto en suelos de climas cálidos y húmedos (`coco`, `banana`, `cafe`).
3. `p` (fósforo): más alto en cultivos que suelen fertilizarse intensivamente, como `maiz` y `manzana`.
4. `cec`: no esperamos grandes diferencias entre cultivos.
5. Si los outliers de una variable están concentrados en un cultivo, esperamos que en el gráfico por clase ese cultivo tenga la caja desplazada hacia arriba y que esos valores **dejen de ser outliers** dentro de su propio cultivo.


In [ ]:
filas = []
for col in num_cols:
    mascara = outliers_iqr(df[col])
    if mascara.sum() == 0:
        continue
    reparto = df.loc[mascara, 'clase'].value_counts(normalize=True)
    filas.append({'variable': col,
                  'outliers': mascara.sum(),
                  '% del dataset': round(100 * mascara.mean(), 2),
                  'clase más frecuente': reparto.index[0],
                  '% de los outliers en esa clase': round(100 * reparto.iloc[0], 1)})

outliers_por_clase = pd.DataFrame(filas).sort_values('% de los outliers en esa clase', ascending=False)
outliers_por_clase

Si los outliers fueran errores al azar, cada clase concentraría alrededor del 10 % de ellos. Graficamos por clase las variables con más de 1000 outliers y alta concentración en un cultivo (`cec` 67 %, `lluvia` 60 %, `soc` 43 %, `p` 34 %):

In [ ]:
vars_por_clase = ['lluvia', 'cec', 'soc', 'p']
orden_clases = sorted(df['clase'].unique())

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
for ax, col in zip(axes.flatten(), vars_por_clase):
    sns.boxplot(data=df, x='clase', y=col, order=orden_clases, ax=ax)
    ax.set_title(col)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)
plt.suptitle('Distribución por clase de las variables con outliers más concentrados', fontsize=16)
plt.tight_layout()
plt.show()

**Observación:**

1. `lluvia`: `coco` tiene la mediana más alta (≈ 2300 mm), seguido de `cafe`, `banana` y `arroz` (≈ 1450–1600 mm); `lenteja` tiene la caja más baja y angosta (≈ 400 mm). **Se cumple.**
2. `soc`: `coco` tiene la caja más alta y una cola que llega a 320; `arroz` y `banana` también tienen muchos valores altos. **Se cumple.**
3. `p`: las cajas más altas y anchas son las de `maiz` y `manzana`, y `maiz` concentra los valores extremos (hasta 180). **Se cumple.**
4. `cec`: **no se cumple.** `garbanzo` se separa claramente del resto (mediana ≈ 29 y caja mucho más ancha), lo que explica que concentre el 67 % de los outliers globales de `cec`.
5. **Se cumple en parte.** En `cec` y `lluvia`, muchos de los valores que en el boxplot global eran outliers son el rango normal de un cultivo. En `cec`, la parte alta de la caja de `garbanzo` y su bigote (hasta ≈ 55) superan el bigote global (≈ 37) sin ser outliers dentro de su cultivo. Lo mismo pasa con `coco` en `lluvia`, entre ≈ 3200 y 4700 mm. En `soc` y `p`, en cambio, muchos puntos siguen fuera de los bigotes de su propio cultivo: además de diferencias entre cultivos, hay valores extremos dentro de cada uno. Recortarlos al bigote global borraría justamente lo que distingue a esos cultivos.


**Decisión: no tratar los outliers IQR.** Una vez eliminados los valores imposibles, los extremos restantes son físicamente posibles y están **concentrados en clases específicas** (tabla y gráfico anteriores): son información útil para clasificar, no ruido. Recortarlos (winsorización) borraría parte de esa señal y, además, si los límites se calculan con todo el dataset antes de la validación cruzada, el fold de validación influiría en la transformación del entrenamiento.

### 1.4 Estandarización de las características numéricas

El `StandardScaler` se incluye **dentro de un `Pipeline`** junto con el `SVC`. Así, en cada fold de la validación cruzada el escalador se ajusta solo con los datos de entrenamiento de ese fold y luego se aplica al de validación, sin que la validación influya en la media y el desvío usados.


In [ ]:
X = df[num_cols]

def pipeline_svm(kernel):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('svc', SVC(kernel=kernel, random_state=SEMILLA)),
    ])

# Solo para verificar el efecto del escalado (no se usa para entrenar): media ≈ 0 y desvío ≈ 1.
X_escalado = StandardScaler().fit_transform(X)
pd.DataFrame(X_escalado, columns=num_cols).describe().loc[['mean', 'std']].round(3)

### 1.5 Codificación de la variable objetivo

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df['clase'])

pd.DataFrame({'clase': le.classes_, 'código': range(len(le.classes_))}).set_index('código').T

## 2. SVM con kernel lineal

> **Consigna:** predecir `clase` con `SVC(kernel='linear')`, explorando `C` con **al menos 5 valores en escala logarítmica**. Validación cruzada estratificada `k=5` sobre todo el conjunto limpio.

### 2.1 Configuración de la validación cruzada

In [ ]:
# shuffle=True es necesario: el CSV viene ordenado por clase.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

SCORING = {
    'accuracy': 'accuracy',
    'precision_weighted': 'precision_weighted',
    'recall_weighted': 'recall_weighted',
}

# Los resultados de cada grilla se guardan en disco para no reentrenar al re-ejecutar.
# Poner RECALCULAR = True para forzar el reentrenamiento.
DIR_RESULTADOS = Path('resultados')
DIR_RESULTADOS.mkdir(exist_ok=True)
RECALCULAR = False

# Procesos en paralelo para las grillas de SVM (cada SVC usa un núcleo).
# Bajarlo si la computadora se queda sin memoria.
N_JOBS_SVM = 8


def correr_grilla(nombre, estimador, grilla, n_jobs):
    ruta = DIR_RESULTADOS / f'{nombre}.joblib'
    if ruta.exists() and not RECALCULAR:
        print(f'Resultados cargados desde {ruta}')
        return joblib.load(ruta)

    busqueda = GridSearchCV(estimador, grilla, scoring=SCORING, cv=skf,
                            refit=False, return_train_score=True,
                            n_jobs=n_jobs, verbose=1)
    inicio = time.perf_counter()
    busqueda.fit(X, y)
    print(f'Tiempo total de la grilla: {(time.perf_counter() - inicio) / 60:.1f} min')

    resultados = pd.DataFrame(busqueda.cv_results_)
    joblib.dump(resultados, ruta)
    return resultados


def tabla_resultados(res, params):
    """Media ± desvío de los 5 folds para cada métrica, más la brecha train-validación."""
    tabla = pd.DataFrame({p.split('__')[-1]: res[f'param_{p}'] for p in params})
    for metrica, nombre in [('accuracy', 'Accuracy'),
                            ('precision_weighted', 'Precision (weighted)'),
                            ('recall_weighted', 'Recall (weighted)')]:
        tabla[nombre] = [f'{m:.4f} ± {s:.4f}'
                         for m, s in zip(res[f'mean_test_{metrica}'], res[f'std_test_{metrica}'])]
    tabla['Accuracy train'] = res['mean_train_accuracy'].round(4)
    tabla['Brecha train-val'] = (res['mean_train_accuracy'] - res['mean_test_accuracy']).round(4)
    tabla['Ajuste por fold (s)*'] = res['mean_fit_time'].round(1)
    return tabla


def mejor_configuracion(res, params):
    """Imprime la mejor configuración y las que quedan a menos de 1 desvío de ella."""
    i = res['mean_test_accuracy'].idxmax()
    mejor, desvio = res.loc[i, 'mean_test_accuracy'], res.loc[i, 'std_test_accuracy']
    print('Mejor configuración:', {p.split('__')[-1]: valor_python(p, res.loc[i, f'param_{p}']) for p in params})
    print(f'  Accuracy  {mejor:.4f} ± {desvio:.4f}')
    print(f'  Precision {res.loc[i, "mean_test_precision_weighted"]:.4f} ± {res.loc[i, "std_test_precision_weighted"]:.4f}')
    print(f'  Recall    {res.loc[i, "mean_test_recall_weighted"]:.4f} ± {res.loc[i, "std_test_recall_weighted"]:.4f}')
    cercanas = res[res['mean_test_accuracy'] >= mejor - desvio]
    print(f'\nConfiguraciones a menos de 1 desvío de la mejor: {len(cercanas)}')
    print(tabla_resultados(cercanas, params).to_string(index=False))
    return {p: valor_python(p, res.loc[i, f'param_{p}']) for p in params}


def valor_python(param, v):
    """Convierte el valor guardado en cv_results_ al tipo que espera sklearn (None, int o float)."""
    if v is None or pd.isna(v):
        return None
    return int(v) if param in ('n_estimators', 'max_depth') else float(v)

\* El tiempo de ajuste por fold es **referencial**: las configuraciones se entrenan en paralelo y compiten por el procesador. La comparación de tiempos entre modelos se hace en la sección 5.1, entrenando cada uno por separado.

### 2.2 Exploración del parámetro `C`

**Hipótesis:** con 10 cultivos de requerimientos superpuestos, esperamos que un modelo lineal logre un rendimiento **moderado** (lejos de 1) y con un techo claro. Al aumentar `C` desde 0,01, la accuracy debería mejorar al principio y después **amesetarse**, mientras el tiempo de entrenamiento crece mucho para los `C` altos. Como el modelo es simple, esperamos una brecha train–validación muy chica. Esperamos que accuracy, precision y recall *weighted* den valores muy parecidos.


In [ ]:
grilla_lineal = {'svc__C': [0.01, 0.1, 1, 10, 100]}

res_lineal = correr_grilla('svm_lineal', pipeline_svm('linear'), grilla_lineal, n_jobs=N_JOBS_SVM)

### 2.3 Resultados

In [ ]:
tabla_resultados(res_lineal, list(grilla_lineal))

In [ ]:
C_vals = res_lineal['param_svc__C'].astype(float)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].errorbar(C_vals, res_lineal['mean_test_accuracy'], yerr=res_lineal['std_test_accuracy'],
                 marker='o', capsize=4, label='Accuracy validación (media ± desvío)')
axes[0].plot(C_vals, res_lineal['mean_train_accuracy'], marker='s', linestyle='--', label='Accuracy entrenamiento')
axes[0].plot(C_vals, res_lineal['mean_test_precision_weighted'], marker='^', alpha=0.7, label='Precision weighted validación')
axes[0].plot(C_vals, res_lineal['mean_test_recall_weighted'], marker='v', alpha=0.7, label='Recall weighted validación')
axes[0].set_xscale('log')
axes[0].set_xlabel('C (escala logarítmica)')
axes[0].set_ylabel('Métrica')
axes[0].set_title('Métricas en función de C')
axes[0].legend()

axes[1].plot(C_vals, res_lineal['mean_fit_time'], marker='o', color='tab:red')
axes[1].set_xscale('log')
axes[1].set_xlabel('C (escala logarítmica)')
axes[1].set_ylabel('Segundos')
axes[1].set_title('Tiempo medio de ajuste por fold (referencial*)')

plt.suptitle('SVM lineal: efecto de C', fontsize=16)
plt.tight_layout()
plt.show()


**Observación:** la accuracy de validación pasa de 0,6047 (`C` = 0,01) a 0,6178 (`C` = 0,1) y desde ahí casi no cambia: 0,6222, 0,6226 y 0,6232. Entre `C` = 1 y `C` = 100 la diferencia (0,001) es mucho menor que el desvío entre folds (≈ 0,006). La brecha train–validación es mínima en todos los casos (≤ 0,0023): el modelo no sobreajusta, su límite es que una frontera lineal no alcanza para separar las clases (subajuste). El recall *weighted* coincide exactamente con la accuracy en todas las configuraciones (también en las grillas del RBF y del Random Forest y en la tabla de 5.1), por lo que en adelante comparamos con accuracy y precision. En este modelo la precision difiere de la accuracy en milésimas. El tiempo por fold crece de 56 s (`C` = 0,1) a 1037 s (`C` = 100), unas 18 veces. **Se cumple la hipótesis:** rendimiento moderado, meseta desde `C` = 1, brecha casi nula y costo creciente con `C`.


### 2.4 Selección de la mejor configuración y justificación

In [ ]:
mejor_lineal = mejor_configuracion(res_lineal, list(grilla_lineal))

**Justificación:** la mejor accuracy media la obtiene `C` = 100 (0,6232 ± 0,0058), aunque `C` = 0,1, 1 y 10 quedan a menos de un desvío: prácticamente son equivalentes. Que el mejor valor esté en el borde de la grilla no indica que convenga seguir aumentando `C`, porque la curva ya está plana. Usamos `C` = 100 para la comparación de la actividad 5 por ser el de mejor media, pero aclaramos que `C` = 1 daría el mismo resultado práctico con un tiempo de entrenamiento unas 12 veces menor (87 s contra 1037 s por fold). En cualquier caso, el modelo lineal queda muy por debajo de los no lineales, así que esta elección no cambia la conclusión final.


## 3. SVM con kernel gaussiano (RBF)

> **Consigna:** predecir `clase` con `SVC(kernel='rbf')`, explorando `C` y `gamma` con **al menos 4 valores cada uno en escala logarítmica**. Validación cruzada estratificada `k=5`.

### 3.1 Exploración de `C` y `gamma`

**Hipótesis:**

1. El kernel RBF va a superar claramente al lineal, porque cada cultivo ocupa rangos acotados de varias variables a la vez (combinaciones no lineales).
2. Con `gamma` = 0,001 el modelo va a subajustar y dar resultados parecidos al SVM lineal.
3. El mejor desempeño estará en valores intermedios de `gamma` (0,01–0,1) con `C` alto (10–100).
4. Con `gamma` = 1 la accuracy de validación va a caer por sobreajuste, con la mayor brecha train–validación de la grilla.


In [ ]:
grilla_rbf = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': [0.001, 0.01, 0.1, 1],
}

res_rbf = correr_grilla('svm_rbf', pipeline_svm('rbf'), grilla_rbf, n_jobs=N_JOBS_SVM)

### 3.2 Resultados

In [ ]:
tabla_resultados(res_rbf, list(grilla_rbf))

In [ ]:
tabla_rbf = res_rbf.assign(C=res_rbf['param_svc__C'].astype(float),
                           gamma=res_rbf['param_svc__gamma'].astype(float),
                           brecha=res_rbf['mean_train_accuracy'] - res_rbf['mean_test_accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(tabla_rbf.pivot(index='C', columns='gamma', values='mean_test_accuracy'),
            annot=True, fmt='.4f', cmap='Blues', ax=axes[0])
axes[0].set_title('Accuracy de validación (media de 5 folds)')
sns.heatmap(tabla_rbf.pivot(index='C', columns='gamma', values='brecha'),
            annot=True, fmt='.4f', cmap='Reds', ax=axes[1])
axes[1].set_title('Brecha accuracy train − validación')
for ax in axes:
    ax.invert_yaxis()
plt.suptitle('SVM RBF: interacción entre C y gamma', fontsize=16)
plt.tight_layout()
plt.show()

**Observación:**

1. El mejor RBF llega a 0,8853, contra 0,6232 del mejor lineal: 26 puntos más. **Se cumple.**
2. Con `gamma` = 0,001 la accuracy va de 0,519 a 0,689 según `C`, del mismo orden que el modelo lineal (≈ 0,62). **Se cumple.**
3. **Se cumple parcialmente.** Los mejores resultados están con `C` alto, pero con `gamma` entre 0,1 y 1, no entre 0,01 y 0,1: con `gamma` = 0,01 la accuracy no supera 0,81. La zona buena forma una **diagonal**: a medida que crece `C` conviene un `gamma` algo menor.
4. **No se cumple del todo.** La brecha sí es máxima con `gamma` = 1 (0,106 con `C` = 10 y 0,119 con `C` = 100, con accuracy de entrenamiento de 0,99–0,998), pero la validación solo cae en la combinación más compleja (`C` = 100, `gamma` = 1: 0,8792). Con `C` = 10 y `gamma` = 1 la validación (0,8851) es prácticamente igual a la mejor. El sobreajuste aparece, pero es moderado.

La mejor configuración (`C` = 100, `gamma` = 0,1) está en el borde de la grilla en `C`. Sin embargo, siguiendo la diagonal de la zona buena, (1, 1) → (10, 1) → (100, 0,1), la accuracy pasa de 0,881 a 0,885 y 0,885: la ganancia ya se agotó y la brecha ya llega a 0,066, por lo que no esperamos una mejora importante ampliando la grilla.


### 3.3 Selección de la mejor configuración y justificación

In [ ]:
mejor_rbf = mejor_configuracion(res_rbf, list(grilla_rbf))

**Justificación:** quedan dos configuraciones a menos de un desvío: `C` = 100, `gamma` = 0,1 (0,8853 ± 0,0022) y `C` = 10, `gamma` = 1 (0,8851 ± 0,0048). Elegimos **`C` = 100, `gamma` = 0,1** porque, con la misma accuracy, tiene la **mitad del desvío** entre folds (más estable), **menor brecha** train–validación (0,066 contra 0,106) y tarda aproximadamente la mitad por fold (48 s contra 93 s). Por precision *weighted* ganaría `C` = 10, `gamma` = 1 (0,8867 contra 0,8843), pero esa diferencia también está dentro del desvío.


## 4. Random Forest

> **Consigna:** predecir `clase` con `RandomForestClassifier`, explorando `n_estimators` (≥ 4 valores) y `max_depth` (≥ 4 valores). Validación cruzada estratificada `k=5`.

### 4.1 Exploración de `n_estimators` y `max_depth`

**Hipótesis:** `max_depth` va a influir mucho más que `n_estimators`: árboles de profundidad 5 (como máximo 32 hojas) no alcanzan para separar 10 cultivos, y esperamos un salto grande al pasar a 20 o `None`. Aumentar `n_estimators` por encima de 100–200 debería aportar mejoras marginales en accuracy, reducir el desvío entre folds y aumentar el tiempo de entrenamiento en forma aproximadamente lineal. Con `max_depth` = `None` esperamos accuracy de entrenamiento cercana a 1, es decir, una brecha train–validación grande.


In [ ]:
grilla_rf = {
    'n_estimators': [50, 100, 200, 500],
    'max_depth': [5, 10, 20, None],
}

# El paralelismo se hace dentro de cada bosque (n_jobs=-1) y la grilla corre de a una
# configuración, para no tener muchos bosques grandes en memoria al mismo tiempo.
rf_base = RandomForestClassifier(random_state=SEMILLA, n_jobs=-1)
res_rf = correr_grilla('random_forest', rf_base, grilla_rf, n_jobs=1)

### 4.2 Resultados

In [ ]:
tabla_resultados(res_rf, list(grilla_rf))

In [ ]:
tabla_rf = res_rf.assign(
    n_estimators=res_rf['param_n_estimators'].astype(int),
    max_depth=res_rf['param_max_depth'].map(lambda v: 'None' if pd.isna(v) else str(int(v))),
    brecha=res_rf['mean_train_accuracy'] - res_rf['mean_test_accuracy'])
orden_prof = ['5', '10', '20', 'None']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, valor, titulo, cmap in [(axes[0], 'mean_test_accuracy', 'Accuracy de validación (media de 5 folds)', 'Greens'),
                                (axes[1], 'brecha', 'Brecha accuracy train − validación', 'Reds')]:
    pivot = tabla_rf.pivot(index='max_depth', columns='n_estimators', values=valor).loc[orden_prof]
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap=cmap, ax=ax)
    ax.set_title(titulo)
plt.suptitle('Random Forest: interacción entre max_depth y n_estimators', fontsize=16)
plt.tight_layout()
plt.show()

**Observación:**

* **`max_depth` domina:** el heatmap cambia de color por filas, no por columnas. La accuracy pasa de ≈ 0,64 (profundidad 5) a ≈ 0,84 (10) y ≈ 0,91 (20 y `None`). Entre 20 y `None` la diferencia ya es menor a un desvío. **Se cumple.**
* **`n_estimators` aporta poco:** con `max_depth` = `None`, pasar de 50 a 500 árboles mejora de 0,9123 a 0,9157, y desde 200 la curva es plana (0,9158 contra 0,9157). **Se cumple.** En cambio, el desvío entre folds **no** baja de forma sistemática (0,0028, 0,0025, 0,0029, 0,0027). **Esa parte no se cumple:** la variabilidad entre folds depende más de qué registros caen en cada fold que de la cantidad de árboles.
* **Tiempo:** crece de forma casi proporcional al número de árboles (1,5 → 2,9 → 5,4 → 12,4 s por fold con `max_depth` = `None`). **Se cumple.**
* **Brecha:** con `max_depth` = `None` la accuracy de entrenamiento es 1,000 y la brecha ≈ 0,084; con profundidad 20, ≈ 0,081. **Se cumple.** Aun así, esas configuraciones son las de **mejor validación**: la brecha grande no implica un peor modelo: aunque el bosque acierta el 100 % del entrenamiento, también logra la mejor validación de toda la grilla.


### 4.3 Importancia de las variables del mejor modelo

**Hipótesis:**

1. Las variables **climáticas** (`lluvia`, las tres temperaturas, `humedad`, `estacionalidad_lluvias`) van a dominar la importancia por sobre las de **suelo** (`ph`, `arcilla`, `arena`, `n`, `p`, `soc`), porque el clima define primero dónde se puede cultivar cada especie. Esto también pone a prueba la sospecha planteada en 1.2.1.
2. Como las tres temperaturas están muy correlacionadas, esperamos que la importancia se **reparta** entre ellas en lugar de concentrarse en una sola.
3. La accuracy OOB debería ser parecida a la de validación cruzada.


In [ ]:
mejor_rf = mejor_configuracion(res_rf, list(grilla_rf))

# Reentrenamos el mejor bosque con todo el conjunto limpio para analizar sus importancias
# y su error out-of-bag (OOB).
rf_final = RandomForestClassifier(**mejor_rf, oob_score=True, random_state=SEMILLA, n_jobs=-1)
rf_final.fit(X, y)

acc_cv = res_rf.loc[res_rf['mean_test_accuracy'].idxmax(), 'mean_test_accuracy']
print(f'\nAccuracy OOB del mejor bosque:    {rf_final.oob_score_:.4f}')
print(f'Accuracy de validación cruzada:   {acc_cv:.4f}')

In [ ]:
importancias = pd.DataFrame({'features': num_cols, 'importances': rf_final.feature_importances_})
importancias = importancias.sort_values(by='importances', ascending=False)

plt.figure(figsize=(10, 7))
g = sns.barplot(data=importancias, x='importances', y='features')
sns.despine(bottom=True, left=True)
g.set_title('Importancia de las características (Random Forest)')
g.set(xlabel=None, ylabel=None, xticks=[])
for contenedor in g.containers:
    g.bar_label(contenedor, fmt='%.3f', padding=2)
plt.tight_layout()
plt.show()

**Observación:**

1. Las 7 variables más importantes son todas climáticas: `temp_min_mensual` (0,113), `temp_max_mensual` (0,090), `lluvia` (0,090), `radiacion_solar` (0,088), `temperatura` (0,085), `humedad` (0,085) y `estacionalidad_lluvias` (0,077). Les sigue `elevacion` (0,072). Juntas suman ≈ 0,70; las 8 de suelo suman ≈ 0,30, y la mejor de ellas, `ph`, apenas llega a 0,045. `p` (0,040) y `soc` (0,037) quedan entre las últimas. **Se cumple**, y confirma la sospecha de 1.2.1.
2. Las tres temperaturas reparten su importancia (0,113 + 0,090 + 0,085). Ninguna se lleva todo, a pesar de que `temperatura` y `temp_min_mensual` tienen r = 0,95. **Se cumple.**
3. Accuracy OOB = 0,9213 contra 0,9158 de validación cruzada. **Se cumple:** son parecidas. La OOB es levemente mayor porque cada árbol del bosque final se entrena con muestras de los 49 789 registros, y no con el 80 % como en cada fold.

**No previsto:** `cec`, que en 1.3.4 separaba claramente a `garbanzo`, tiene una importancia baja (0,038). Una variable que distingue a un solo cultivo de diez aporta poco a la importancia global. Por eso una importancia baja no significa que la variable sea inútil.


### 4.4 Selección de la mejor configuración y justificación

**Justificación:** la mejor configuración es **`n_estimators` = 200, `max_depth` = `None`** (0,9158 ± 0,0029). Otras cuatro quedan a menos de un desvío, entre ellas 100 árboles sin límite de profundidad (0,9142) y 200 árboles con profundidad 20 (0,9142). Elegimos la de mejor media porque su costo es moderado (≈ 5 s por fold). Pasar a 500 árboles no mejora (0,9157) y cuesta más del doble de tiempo. La brecha train–validación (0,084) no es preocupante: la validación es la mejor de toda la grilla y la estimación OOB (0,921) la confirma.


## 5. Comparación de modelos

### 5.1 Tabla comparativa

Las métricas son las de validación cruzada (media ± desvío de los 5 folds). El **tiempo de entrenamiento** se mide reentrenando cada mejor modelo, por separado y con **un solo núcleo**, sobre el 80 % de entrenamiento del split 80/20 estratificado que se usa para las matrices de confusión (5.2).

**Hipótesis:** esperamos que el Random Forest tenga la mayor accuracy, seguido de cerca por el SVM RBF y muy por debajo el SVM lineal, con barras de desvío tan chicas que no se superpongan entre modelos. En tiempo de entrenamiento esperamos que el SVM lineal (con `C` = 100) sea el más lento por amplia diferencia, y que el RBF y el Random Forest estén en el mismo orden de magnitud.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA)

modelos = {
    'SVM lineal': (pipeline_svm('linear').set_params(**mejor_lineal), res_lineal),
    'SVM RBF': (pipeline_svm('rbf').set_params(**mejor_rbf), res_rbf),
    'Random Forest': (RandomForestClassifier(**mejor_rf, random_state=SEMILLA, n_jobs=1), res_rf),
}

filas, predicciones = [], {}
for nombre, (modelo, res) in modelos.items():
    inicio = time.perf_counter()
    modelo.fit(X_train, y_train)
    t_entrenamiento = time.perf_counter() - inicio
    predicciones[nombre] = modelo.predict(X_test)

    i = res['mean_test_accuracy'].idxmax()
    filas.append({
        'Modelo': nombre,
        'Hiperparámetros': {k.split('__')[-1]: v for k, v in modelo.get_params().items()
                            if k in ('svc__C', 'svc__gamma', 'n_estimators', 'max_depth')
                            and not (nombre == 'SVM lineal' and k == 'svc__gamma')},
        'Accuracy (CV)': f"{res.loc[i, 'mean_test_accuracy']:.4f} ± {res.loc[i, 'std_test_accuracy']:.4f}",
        'Precision w (CV)': f"{res.loc[i, 'mean_test_precision_weighted']:.4f} ± {res.loc[i, 'std_test_precision_weighted']:.4f}",
        'Recall w (CV)': f"{res.loc[i, 'mean_test_recall_weighted']:.4f} ± {res.loc[i, 'std_test_recall_weighted']:.4f}",
        'Brecha train-val': round(res.loc[i, 'mean_train_accuracy'] - res.loc[i, 'mean_test_accuracy'], 4),
        'Accuracy test 20 %': round(accuracy_score(y_test, predicciones[nombre]), 4),
        'Entrenamiento (s, 1 núcleo)': round(t_entrenamiento, 1),
    })

comparacion = pd.DataFrame(filas).set_index('Modelo')
comparacion

In [ ]:
nombres = list(modelos)
mejores = [res.loc[res['mean_test_accuracy'].idxmax()] for _, res in modelos.values()]
colores = sns.color_palette(n_colors=len(nombres))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

medias = [m['mean_test_accuracy'] for m in mejores]
axes[0].bar(nombres, medias, yerr=[m['std_test_accuracy'] for m in mejores], capsize=8, color=colores)
axes[0].set_ylim(0.5, 1)
axes[0].set_ylabel('Accuracy (media ± desvío de 5 folds)')
axes[0].set_title('Accuracy de validación cruzada')
for x, valor in enumerate(medias):
    axes[0].text(x, valor + 0.015, f'{valor:.4f}', ha='center')

tiempos = comparacion.loc[nombres, 'Entrenamiento (s, 1 núcleo)']
axes[1].bar(nombres, tiempos, color=colores)
axes[1].set_yscale('log')
axes[1].set_ylabel('Segundos (escala logarítmica)')
axes[1].set_title('Tiempo de entrenamiento (1 núcleo, 80 % de los datos)')
for x, valor in enumerate(tiempos):
    axes[1].text(x, valor * 1.1, f'{valor:.1f} s', ha='center')

plt.suptitle('Comparación de los mejores modelos', fontsize=16)
plt.tight_layout()
plt.show()


**Observación:** el orden de accuracy es el esperado: Random Forest (0,9158), SVM RBF (0,8853) y SVM lineal (0,6232). Las barras de desvío son tan chicas frente a las diferencias entre modelos que no se superponen. **Se cumple.** En tiempo, el SVM lineal tarda 572 s, unas 18 veces más que el RBF (31,9 s) y 15 veces más que el Random Forest (37,2 s), mientras que estos dos están en el mismo orden de magnitud. **Se cumple.** Los tiempos exactos varían entre ejecuciones según la carga de la computadora, pero el orden entre modelos y la gran diferencia del lineal se mantienen.


### 5.2 Matrices de confusión

**Hipótesis:**

1. Los errores no se van a repartir de forma uniforme: se van a concentrar entre cultivos con requerimientos parecidos, como las legumbres (`garbanzo`, `frijol`, `lenteja`) o los tropicales (`cafe`, `banana`, `coco`).
2. Los pares con etiquetas contradictorias encontrados en 1.3.2 (`garbanzo`–`lenteja` y `maiz`–`manzana`) van a estar entre los más confundidos.
3. Los modelos no lineales (RBF y Random Forest) van a reducir mucho los errores del lineal, aunque los tres se van a equivocar más o menos en los mismos pares.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(27, 8))
for ax, (nombre, y_pred) in zip(axes, predicciones.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=le.classes_, cmap=plt.cm.Blues,
        xticks_rotation=45, colorbar=False, ax=ax)
    ax.set_title(f'{nombre} (accuracy test = {accuracy_score(y_test, y_pred):.3f})')
plt.suptitle('Matrices de confusión de los mejores modelos (split 80/20, random_state=13)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Pares de clases más confundidos por cada modelo (sumando ambos sentidos A→B y B→A),
# marcando los que también aparecían con etiquetas contradictorias en el dataset (1.3.2).
filas = []
for nombre, y_pred in predicciones.items():
    cm = confusion_matrix(y_test, y_pred)
    for i in range(len(le.classes_)):
        for j in range(i + 1, len(le.classes_)):
            par = frozenset((le.classes_[i], le.classes_[j]))
            filas.append({'Modelo': nombre,
                          'Par': f'{le.classes_[i]} ↔ {le.classes_[j]}',
                          'Errores': cm[i, j] + cm[j, i],
                          'Etiqueta contradictoria en el dataset': par in pares_conflictivos})

top_confusiones = (pd.DataFrame(filas).sort_values(['Modelo', 'Errores'], ascending=[True, False])
                   .groupby('Modelo').head(3).set_index(['Modelo', 'Par']))
top_confusiones

**Observación:**

1. En el modelo lineal los errores están muy dispersos, pero los más grandes son entre tropicales: `banana`↔`cafe` (385), `cafe`↔`coco` (233). En el RBF y el Random Forest los errores se concentran en pocos pares: `arroz`↔`maiz`, `arroz`↔`coco` y `banana`↔`cafe`. **Se cumple para los tropicales.** Para las legumbres, solo en parte: `garbanzo`↔`lenteja` suma 44 errores en el Random Forest, pero `frijol` se confunde más con `maiz` y `algodon` que con las otras legumbres.
2. **Se cumple en parte:** `maiz`↔`manzana` es el tercer par más confundido del Random Forest (59), pero `garbanzo`↔`lenteja` no aparece en el top 3 de ningún modelo. Como esas filas se eliminaron en 1.3.2, no son la causa de estos errores: lo que muestran es que esos cultivos comparten condiciones de suelo y clima.
3. Los errores bajan de ≈ 3700 (lineal) a ≈ 1140 (RBF) y ≈ 830 (Random Forest) sobre ≈ 9960 registros de test. **Se cumple** la reducción, pero **no** que los tres fallen en los mismos pares: `banana`↔`cafe` y `algodon`↔`frijol`, muy confundidos por el lineal, dejan de estar entre los más confundidos del Random Forest (bajan de 385 a 51 y de 234 a 30), aunque `banana`↔`cafe` sigue segundo en el RBF (84). Esas clases se pueden separar, pero con una frontera no lineal.

**No previsto:** `maiz` es la clase peor clasificada por el RBF y el Random Forest (en este último acierta 794 de 999) y se confunde con casi todas; en el lineal queda prácticamente empatada con `banana` (445 de 999 contra 442 de 1000). Es un cultivo que se siembra en climas muy variados, por lo que sus registros se superponen con los de muchos otros. Además, la matriz no es simétrica: por ejemplo, el Random Forest confunde 38 `garbanzo` como `lenteja`, pero solo 6 `lenteja` como `garbanzo`.


### 5.3 Selección del modelo más adecuado

Criterios: métricas obtenidas, estabilidad entre folds (desviación estándar), interpretabilidad y tiempo de entrenamiento.

**Justificación:** elegimos el **Random Forest** (`n_estimators` = 200, `max_depth` = `None`).

* **Métricas:** es el mejor en las tres métricas (accuracy 0,9158 contra 0,8853 del RBF y 0,6232 del lineal) y también en el test 80/20 (0,917). La diferencia con el RBF (≈ 0,03) es unas diez veces mayor que el desvío entre folds, así que no es producto del azar.
* **Estabilidad:** los dos modelos no lineales son muy estables (desvío 0,0029 el Random Forest y 0,0022 el RBF). La diferencia entre ellos es despreciable frente a la diferencia de accuracy.
* **Interpretabilidad:** el Random Forest permite medir la importancia de cada variable (4.3), mientras que el SVM RBF no ofrece una forma directa de explicar qué variables usa. El más interpretable sería el lineal, por sus pesos por variable, pero con 0,62 de accuracy no es una opción.
* **Tiempo de entrenamiento** (un núcleo, 80 % de los datos): Random Forest 37,2 s, RBF 31,9 s y lineal 572 s (con `C` = 100; con `C` = 1, que rinde prácticamente igual, tarda bastante menos, aunque sigue siendo el más lento). El Random Forest es algo más lento que el RBF, pero del mismo orden. Además, se puede paralelizar entre árboles y no necesita escalado. El lineal es a la vez el peor y el más lento.

Si el único criterio fuera el tiempo, el RBF sería una alternativa razonable. Pero el Random Forest gana en métricas e interpretabilidad, que son más importantes para un sistema de recomendación de cultivos, donde el entrenamiento se hace una sola vez.


## 6. Conclusiones

**Calidad de los datos.** El dataset tenía errores introducidos a propósito, y los detectamos con código antes de entrenar:

* 1 fila con 3 valores faltantes.
* 198 duplicados exactos, concentrados en `lenteja` y `manzana`.
* 7 puntos con las mismas variables y distinta clase (14 filas).
* 2 filas con valores físicamente imposibles, detectadas con reglas de dominio. Una sola de ellas (la 30012) violaba 6 de las 10 reglas.

En total eliminamos 215 registros (0,43 %). El balance inicial ya daba una pista: `cafe` y `manzana`, dos de las clases con registros de más respecto de 5000, eran las que tenían las filas con errores. Los outliers del criterio IQR **no** se trataron: están concentrados en cultivos específicos (por ejemplo, el 67 % de los de `cec` son de `garbanzo` y el 60 % de los de `lluvia` son de `coco`), así que son información para clasificar y no ruido. La comparación entre boxplots y reglas de dominio mostró que un criterio estadístico como el IQR no alcanza para separar un valor raro de uno imposible.

**Hipótesis.** La mayoría se cumplieron total o parcialmente. Las que no se cumplieron dejaron aprendizajes:

* En el RBF, `gamma` = 1 no empeoró la validación salvo con `C` = 100.
* Más árboles en el Random Forest no redujeron el desvío entre folds.
* `cec` separa muy bien a `garbanzo`, aunque su importancia global es baja.
* Los errores del modelo lineal entre `banana` y `cafe` no eran inevitables: los modelos no lineales los reducen mucho.

Las dos predicciones de "Para lo que viene" (1.2.1) se pueden contrastar ahora:

* **Clima por sobre suelo:** se confirma. Las 7 variables más importantes del Random Forest son climáticas, y `p` y `soc` quedan entre las últimas.
* **SVM más afectado que el Random Forest por los extremos:** los resultados son coherentes, porque el Random Forest, que no usa escalado, es el mejor modelo y no necesitó ningún tratamiento de outliers. Pero no podemos atribuir la diferencia solo a ese factor, porque los modelos difieren en muchos otros aspectos.

**Modelos.** El SVM lineal tiene un techo de ≈ 0,62 por subajuste: no sobreajusta, pero una frontera lineal no alcanza para separar 10 cultivos. El SVM RBF llega a 0,885 y el Random Forest a 0,916. Elegimos el **Random Forest** (200 árboles, sin límite de profundidad) por tener las mejores métricas, una estabilidad comparable a la del RBF, la posibilidad de interpretar la importancia de las variables y un tiempo de entrenamiento del mismo orden que el RBF y unas 15 veces menor que el lineal con `C` = 100.

**Límite del problema.** Aun el mejor modelo se equivoca en ≈ 8 % de los casos, concentrados en `maiz` y `arroz`. Esos errores se explican por el solapamiento entre cultivos que ya se veía en los boxplots por clase (1.3.4), no por los 14 registros con etiquetas contradictorias, que se eliminaron antes de entrenar. Esos registros sí son un indicio del mismo fenómeno: unas mismas condiciones de suelo y clima pueden servir para más de un cultivo, por lo que una exactitud (accuracy) del 100 % no es esperable. El modelo recomienda un cultivo adecuado, no necesariamente el único posible.


## Referencias

1. Guinness World Records. *Hottest month (single location)* — Death Valley, julio de 2018. https://www.guinnessworldrecords.com/world-records/535251-hottest-month-single-location
2. Weather Underground (Weather Historian). *Warmest places on Earth: average annual temperature* — Dallol, 1960–1966. https://www.wunderground.com/blog/weatherhistorian/warmest-places-on-earth-average-annual-temperature.html
